### Imports

In [1]:
import xarray as xr
import gcsfs
import json
from dask.diagnostics import ProgressBar

from google.cloud import storage
from google.oauth2.credentials import Credentials
import warnings

import sys

sys.path.append("../src/")
sys.path.append("../ocean_emulators_main/")
from ocean_emulators.dataset_validation import ds_input_validate, ds_prediction_validate
from ocean_emulators.postprocessing import post_processor

In [2]:
def post_processor(ds: xr.Dataset, ds_truth: xr.Dataset, ls) -> xr.Dataset:
    """Converts the prediction output to an xarray dataset with the same dimensions/variables as input"""
    # Always run the ds_input_validate in non-deep mode here
    try:
        ds_input_validate(ds_truth, deep=False)
    except ValueError as e:
        raise ValueError(
            f"Checking the input dataset failed with {e}. Please fix those issues before creating a postprocessed dataset."
        )

    # correct swapped dimensions and warn
    if len(ds.x) == 180 and len(ds.y) == 360:
        ds = ds.rename({"x": "x_i", "y": "y_i"}).rename({"x_i": "y", "y_i": "x"})
        warnings.warn(
            "Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream"
        )

    da = ds["__xarray_dataarray_variable__"]
    n_lev = 19
    if set(ls) - {"zos"} == set(["uo", "vo", "thetao", "so"]):
        variables = ["uo", "vo", "thetao", "so"]
    elif set(ls) - {"zos"} == set(["thetao", "so"]):
        variables = ["thetao", "so"]
    elif set(ls) - {"zos"} == set(["uo", "vo"]):
        variables = ["uo", "vo"]
    slices = [slice(i, i + n_lev) for i in range(0, len(variables) * n_lev, n_lev)]
    var_slices = {k: sl for k, sl in zip(variables, slices)}
    variables = {
        k: da.isel(var=sl).rename({"var": "lev"}) for k, sl in var_slices.items()
    }
    variables["zos"] = da.isel(var=-1).squeeze()

    ds_out = xr.Dataset(variables)
    for var in ds_out.data_vars:
        if "lev" in ds_out[var].dims:
            ds_out[var] = ds_out[var].where(ds_truth.wetmask)
        else:
            ds_out[var] = ds_out[var].where(ds_truth.wetmask.isel(lev=0))

    ## attach all coordinates from input
    ds_out = ds_out.assign_coords({co: ds_truth[co] for co in ds_truth.coords})

    return ds_out

## Data to leap

In [3]:
with open(
    "/global/homes/s/suryad/.config/gcloud/application_default_credentials.json"
) as f:  # 🚨 make sure to enter the `.json` file from step 7
    token = json.load(f)
fs = gcsfs.GCSFileSystem(token=token)

#### Define Paths

In [12]:
local_truth_path = "/mnt/home/sd5313/data/Ocean/OM4_5daily_v0.2.1_with_hfds_anom_100_years_10repeat_nojump_3xcc"  # OM4_5daily_v0.2.1.zarr"
local_pred_path = "/mnt/lustre/nyu/sd5313/Ocean_Emulator/Preds/2024-09-10_ConvNextUNetTrain3Dv021Eval3Dhfdsanoms1975Nofastinoutnojump3xccEpochs70Epoch55Years100_10repeat-1996_Train_global_3D_Test_global_3D_all_N_train_0_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_0_rand_seed_1.zarr"
leap_path = "gs://leap-persistent/sd5313/TESTconvnext_long_rollout_Temponly-OM4v0.2.1_eval-OM4v0.2.1"

In [10]:
ds_truth = xr.open_zarr(local_truth_path)
ds_truth = ds_truth.isel(time=slice(3, 7003))
ds = xr.open_zarr(local_pred_path)
ds

<xarray.Dataset>
Dimensions:                        (time: 7000, x: 180, y: 360, var: 39)
Dimensions without coordinates: time, x, y, var
Data variables:
    __xarray_dataarray_variable__  (time, x, y, var) float64 dask.array<chunksize=(44, 23, 45, 10), meta=np.ndarray>

#### Validation and Postprocessing


In [11]:
# ds_raw_prediction_validate(ds)
ds_truth = ds_truth.astype("float32")
ls = [
    "thetao",
    "so",
    "zos",
]  # ['uo', 'vo', 'thetao', 'so', 'zos'], ['thetao', 'so', 'zos']
ds = post_processor(ds, ds_truth, ls)
ds = ds.astype("float32")
ds = ds.chunk({"time": 10, "x": -1, "y": -1, "lev": -1})  # 'lev': -1 might help
ds_prediction_validate(ds)
ds

/tmp/ipykernel_913313/4147618335.py:14: UserWarning: Swapped x and y dimensions detected. Fixing this now, but should be corrected upstream
  warnings.warn(
/mnt/lustre/nyu/sd5313/Ocean_Emulator/notebooks/../ocean_emulators_main/ocean_emulators/dataset_validation.py:51: UserWarning: This checks nothing yet
  warnings.warn("This checks nothing yet")


<xarray.Dataset>
Dimensions:         (time: 7000, y: 180, x: 360, lev: 19, y_b: 181, x_b: 361)
Coordinates:
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
    areacello       (y, x) float64 dask.array<chunksize=(180, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(180, 360), meta=np.ndarray>
    lon             (y, x) float64 dask.array<chunksize=(180, 360), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
    wetmask         (lev, y, x) bool dask.array<chunksize=(10, 180, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * time            (time) datetime64[ns] 1996-01-18 1996-01-23 ... 2010-12-03
Dimensions without coordinates: y_b, x_b
Data variables:
    thetao          (time, y, x, lev) float32 dask.array<chunksize=(10, 180, 360, 10), meta=np.ndarray>
    so              (time, y, x, lev) float32 dask.array<chunksize=(10, 180, 360, 1), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(10, 180, 360), meta=np.ndarray>

In [13]:
mapper = fs.get_mapper(leap_path)

with ProgressBar():
    ds.to_zarr(mapper)

ValueError: Zarr requires uniform chunk sizes except for final chunk. Variable named 'so' has incompatible dask chunks: ((10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10

## Data From Leap

In [5]:
# import an access token
# - option 1: read an access token from a file
with open("token.txt") as f:
    access_token = f.read().strip()

# setup a storage client using credentials
credentials = Credentials(access_token)
fs = gcsfs.GCSFileSystem(token=credentials)

#### Define paths

In [10]:
leap_path = "gs://leap-persistent/jbusecke/ocean-emulators/OM4_5daily_v0.2.1.zarr"
local_path = "/mnt/home/sd5313/data/Ocean/OM4_5daily_v0.2.1.zarr"

In [ ]:
ds = xr.open_dataset(fs.get_mapper("leap_path"), engine="zarr", chunks={})
print(ds)

In [ ]:
# encoding = ds.encoding
# for v in ds.variables:
#     if v in encoding.keys():
#         encoding[v]['compressor']=None
#     else:
#         encoding[v] = {'compressor':None}
# with ProgressBar():
#     ds.to_zarr(local_path, encoding = encoding,
#                 consolidated=True, mode='w')

#### Validation

In [11]:
data = xr.open_zarr(local_path)
ds_input_validate(data)